# Algorithms for massive data Project

## Data loading

In [1]:
import os
import zipfile
os.environ['KAGGLE_USERNAME'] = "xxxx"
os.environ['KAGGLE_KEY'] = "xxxx"
!kaggle datasets download -d Cornell-University/arxiv
with zipfile.ZipFile("arxiv.zip", "r") as zip_ref:
    zip_ref.extractall("arxiv_data")

Dataset URL: https://www.kaggle.com/datasets/Cornell-University/arxiv
License(s): CC0-1.0
arxiv.zip: Skipping, found more recently modified local copy (use --force to force download)


## Configuration

In [ ]:
import json
DATA_FILE = "arxiv_data/arxiv-metadata-oai-snapshot.json"
MAX_RECORDS = 100000
MIN_PAPERS  = 2
DAMPING = 0.85
TOP_N = 20

## Parsing author names

In [6]:
def get_authors(record):
    parsed = record.get("authors_parsed") or []
    if parsed:
        names = []
        for entry in parsed:
            last = entry[0].strip() if len(entry) > 0 else ""
            first = entry[1].strip() if len(entry) > 1 else ""
            if last:
                names.append(f"{last}, {first}" if first else last)
        return names

    return [a.strip() for a in record.get("authors", "").split(",") if a.strip()]

In [ ]:
paper_count = {}

with zipfile.ZipFile("arxiv.zip", 'r') as z:
    with z.open("arxiv-metadata-oai-snapshot.json") as f:
        for line in f:
            record = json.loads(line)
            for author in get_authors(record):
                paper_count[author] = paper_count.get(author, 0) + 1
eligible_authors = {author for author, count in paper_count.items() if count >= MIN_PAPERS}
print(f"Total unique authors: {len(paper_count):,}")
print(f"Eligible authors (written >= {MIN_PAPERS} papers): {len(eligible_authors):,}")

Total unique authors: 2,006,954
Eligible authors (written >= 2 papers): 1,061,552
